# 🦴 Femur 3D Reconstruction from 2D Projections

## Pipeline
```
Femur.stl  →  Voxelize  →  Render 2D projections at [0°, 30°, 60°, 90°]
                                    ↓
                     Apply segmentation method on each 2D image
                     (Otsu / Adaptive / Watershed / Chan-Vese)
                                    ↓
                     Back-project segmented silhouettes → 3D volume
                     (Visual Hull / Shape-from-Silhouette)
                                    ↓
                     Display reconstructed 3D + evaluate vs Ground Truth
```

| Step | Description |
|---|---|
| **Projection** | Rotate STL volume by angle, max-project along Y-axis → 2D silhouette image |
| **Segmentation** | Apply threshold method to isolate bone in each 2D image |
| **Back-projection** | Extrude each segmented 2D mask into 3D, rotate back, intersect all → Visual Hull |
| **Evaluation** | Volume Difference (%) + Dice score vs original STL |

---
## 1 · Imports

In [ ]:
import struct
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from skimage import measure, morphology
from skimage.morphology import ball, closing, dilation
from skimage.filters import threshold_otsu, threshold_local
from skimage.segmentation import watershed, chan_vese
from scipy.ndimage import rotate, gaussian_filter, binary_fill_holes
from scipy import ndimage

warnings.filterwarnings('ignore')
print('All imports OK ✓')

---
## 2 · Load STL & Build Ground-Truth Voxel Volume

In [ ]:
def read_stl_binary(path):
    """Parse binary STL → (N,3,3) triangle array."""
    with open(path, 'rb') as f:
        f.read(80)
        n_tri = struct.unpack('<I', f.read(4))[0]
        tris = []
        for _ in range(n_tri):
            f.read(12)
            tri = [struct.unpack('<fff', f.read(12)) for _ in range(3)]
            f.read(2)
            tris.append(tri)
    return np.array(tris, dtype=np.float32)

def voxelize_surface(triangles, resolution=80):
    """Rasterize STL triangles into a 3D binary voxel grid."""
    verts = triangles.reshape(-1, 3)
    mn, mx = verts.min(0), verts.max(0)
    norm = (verts - mn) / (mx - mn) * (resolution - 3) + 1
    idx  = norm.astype(np.int32).clip(0, resolution - 1)
    grid = np.zeros((resolution,) * 3, dtype=np.float32)
    grid[idx[:, 0], idx[:, 1], idx[:, 2]] = 1.0
    grid = dilation(grid > 0, ball(1)).astype(np.float32)
    return grid, mn, mx

# ── Load ──────────────────────────────────────────────────────
STL_PATH   = 'Femur.stl'   # <-- update path if needed
RESOLUTION = 80
ANGLES     = [0, 30, 60, 90]  # projection angles in degrees

print('Reading STL …')
triangles = read_stl_binary(STL_PATH)
print(f'  {len(triangles):,} triangles')

print('Voxelizing …')
surface_grid, vox_min, vox_max = voxelize_surface(triangles, RESOLUTION)
ground_truth = binary_fill_holes(surface_grid > 0).astype(np.float32)

GT_VOLUME = int(ground_truth.sum())
print(f'  Ground-truth voxels : {GT_VOLUME:,}')
print(f'  Grid shape          : {ground_truth.shape}')
print('Done ✓')

---
## 3 · Step 1 — Render 2D Projection Images at 4 Angles

We rotate the 3D voxel volume around the **Z-axis** (vertical axis of the femur) by each angle, then take a **Maximum Intensity Projection (MIP)** along the Y-axis. This simulates photographing the femur from 4 different directions.

In [ ]:
def render_projection(volume, angle_deg):
    """
    Simulate a 2D 'photograph' of the 3D femur at a given angle.
    
    Steps:
      1. Rotate the volume around the Z-axis by angle_deg
      2. Max-project along Y → 2D silhouette (shape: X × Z)
      3. Apply mild Gaussian blur to simulate realistic image capture
    """
    rotated  = rotate(volume.astype(np.float32), angle_deg,
                      axes=(0, 1), reshape=False, order=1)
    proj     = rotated.max(axis=1)          # max intensity projection → (X, Z)
    proj     = gaussian_filter(proj, sigma=0.8)   # slight blur
    proj     = (proj - proj.min()) / (proj.max() - proj.min() + 1e-9)  # normalize [0,1]
    return proj


# ── Render all 4 projections ──────────────────────────────────
print('Rendering 2D projections from Ground Truth STL …')
projections_gt = {}
for angle in ANGLES:
    projections_gt[angle] = render_projection(ground_truth, angle)
    print(f'  {angle:>2}° → shape {projections_gt[angle].shape},  '
          f'max={projections_gt[angle].max():.3f}')

print('\nDone ✓')

In [ ]:
# ── Display the 4 projection images ──────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 5), facecolor='#0a0a14')
fig.suptitle('Step 1 — 2D Projection Images of Femur STL\n'
             '(Simulated photographs at 0°, 30°, 60°, 90°)',
             color='white', fontsize=13, fontweight='bold', y=1.02)

for ax, angle in zip(axes, ANGLES):
    img = projections_gt[angle]
    ax.imshow(img.T, cmap='bone', origin='lower', aspect='auto')
    ax.set_title(f'Angle = {angle}°', color='#4e9af1',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('X axis', color='#888', fontsize=9)
    ax.set_ylabel('Z axis', color='#888', fontsize=9)
    ax.tick_params(colors='#555')
    for spine in ax.spines.values():
        spine.set_edgecolor('#333')
    ax.set_facecolor('#0a0a14')

plt.tight_layout()
plt.savefig('projections_4angles.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Projection images saved → projections_4angles.png')

---
## 4 · Step 2 — Segmentation Methods on 2D Projection Images

Each method segments the bone from background **in each 2D projection image**, producing a binary silhouette mask.

In [ ]:
# ── Method 1: Otsu ───────────────────────────────────────────
def segment_otsu(image):
    """
    Compute a single global threshold that minimises intra-class
    intensity variance (Otsu, 1979). Clean with morphological closing.
    """
    t      = threshold_otsu(image)
    binary = image > t
    binary = closing(binary, morphology.disk(2))
    binary = morphology.remove_small_objects(binary, max_size=30)
    binary = ndimage.binary_fill_holes(binary)
    return binary.astype(np.float32), t


# ── Method 2: Adaptive Threshold ─────────────────────────────
def segment_adaptive(image, block_size=21, offset=-0.02):
    """
    Local threshold computed in a (block_size × block_size) window.
    Each pixel is compared against its local neighbourhood mean.
    No single global threshold — adapts to local intensity variation.
    """
    local_t = threshold_local(image, block_size=block_size, offset=offset)
    binary  = image > local_t
    binary  = closing(binary, morphology.disk(2))
    binary  = morphology.remove_small_objects(binary, max_size=30)
    binary  = ndimage.binary_fill_holes(binary)
    return binary.astype(np.float32)


# ── Method 3: Watershed ───────────────────────────────────────
def segment_watershed(image):
    """
    Gradient-based watershed with marker seeding.
    Seeds are placed at:
      - high-intensity regions (definite bone)
      - near-zero regions (definite background)
    The algorithm floods the gradient surface from these seeds.
    """
    from skimage.filters import sobel
    from skimage.segmentation import watershed as skwatershed
    
    t       = threshold_otsu(image)
    grad    = sobel(image)
    markers = np.zeros_like(image, dtype=np.int32)
    markers[ndimage.binary_erosion(image > t, iterations=3)] = 1  # bone seeds
    markers[image < t * 0.25] = 2                                  # background seeds
    result  = skwatershed(grad, markers) == 1
    result  = closing(result, morphology.disk(2))
    result  = morphology.remove_small_objects(result, max_size=30)
    result  = ndimage.binary_fill_holes(result)
    return result.astype(np.float32)


# ── Method 4: Chan-Vese ───────────────────────────────────────
def segment_chan_vese(image, mu=0.25, max_iter=200):
    """
    Variational region-energy active contour.
    Minimises: area_inside*(mean_inside - c1)² + area_outside*(mean_outside - c2)²
    + mu * contour_length
    No gradient required — works on low-contrast edges.
    """
    seg = chan_vese(image, mu=mu, lambda1=1.0, lambda2=1.0,
                   tol=1e-4, max_num_iter=max_iter, dt=0.5,
                   init_level_set='checkerboard')
    # Pick the region with higher mean (= bone)
    if image[seg].mean() < image[~seg].mean():
        seg = ~seg
    seg = closing(seg, morphology.disk(2))
    seg = morphology.remove_small_objects(seg, max_size=30)
    seg = ndimage.binary_fill_holes(seg)
    return seg.astype(np.float32)


METHODS = {
    'Otsu'              : segment_otsu,
    'Adaptive Threshold': segment_adaptive,
    'Watershed'         : segment_watershed,
    'Chan-Vese'         : segment_chan_vese,
}

PALETTE = {
    'Ground Truth'      : '#4e9af1',
    'Otsu'              : '#f06292',
    'Adaptive Threshold': '#4caf50',
    'Watershed'         : '#ffb74d',
    'Chan-Vese'         : '#ce93d8',
}

print('Segmentation methods defined ✓')

In [ ]:
# ── Segment all projections with all methods ──────────────────
print('Segmenting projection images …')
segmented = {}   # segmented[method_name][angle] = 2D binary mask
otsu_thresholds = {}

for method_name, method_fn in METHODS.items():
    segmented[method_name] = {}
    print(f'  [{method_name}]')
    for angle in ANGLES:
        img = projections_gt[angle]
        if method_name == 'Otsu':
            mask, t = method_fn(img)
            otsu_thresholds[angle] = t
        else:
            mask = method_fn(img)
        segmented[method_name][angle] = mask
        print(f'    {angle:>2}° → nonzero={mask.sum():.0f}')

print()
print('Otsu threshold values per angle:')
for angle, t in otsu_thresholds.items():
    print(f'  {angle:>2}° → {t:.6f}')
print('\nSegmentation done ✓')

In [ ]:
# ── Display segmented projection images for all methods ───────
cmaps_seg = {
    'Otsu'              : 'RdPu',
    'Adaptive Threshold': 'Greens',
    'Watershed'         : 'Oranges',
    'Chan-Vese'         : 'Purples',
}

fig, axes = plt.subplots(len(METHODS) + 1, len(ANGLES),
                          figsize=(16, 22), facecolor='#0a0a14')
fig.suptitle('Step 2 — Segmented 2D Projection Images\n'
             '(Each row = one method, each column = one angle)',
             color='white', fontsize=13, fontweight='bold', y=1.0)

# Row 0: original projections
for col, angle in enumerate(ANGLES):
    ax = axes[0][col]
    ax.imshow(projections_gt[angle].T, cmap='bone', origin='lower', aspect='auto')
    ax.set_title(f'{angle}°', color='#4e9af1', fontsize=12, fontweight='bold')
    ax.set_axis_off()
    ax.set_facecolor('#0a0a14')
axes[0][0].set_ylabel('Original\nProjection', color='#4e9af1',
                       fontsize=10, fontweight='bold')
axes[0][0].yaxis.set_visible(True)

# Rows 1-4: segmented masks
for row, (method_name, cmap) in enumerate(cmaps_seg.items(), start=1):
    color = PALETTE[method_name]
    for col, angle in enumerate(ANGLES):
        ax = axes[row][col]
        mask  = segmented[method_name][angle]
        orig  = projections_gt[angle]
        # Overlay: original in grey, segmented in colour
        overlay = np.stack([
            orig * 0.4 + mask * 0.6,
            orig * 0.4,
            orig * 0.4,
        ], axis=-1)
        overlay = np.clip(overlay, 0, 1)
        ax.imshow(overlay.transpose(1, 0, 2), origin='lower', aspect='auto')
        ax.set_axis_off()
        ax.set_facecolor('#0a0a14')
    axes[row][0].set_ylabel(method_name, color=color,
                             fontsize=10, fontweight='bold')
    axes[row][0].yaxis.set_visible(True)

for ax_row in axes:
    for ax in ax_row:
        for spine in ax.spines.values():
            spine.set_visible(False)

plt.tight_layout()
plt.savefig('segmented_projections.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Saved → segmented_projections.png')

---
## 5 · Step 3 — Back-Project to Reconstruct 3D

**Visual Hull (Shape-from-Silhouette):** For each segmented 2D mask:
1. Extrude the mask along the projection axis → 3D silhouette cone
2. Rotate the cone back to the original orientation
3. **Intersect** all 4 cones → the reconstruction is the region consistent with all views

More angles = tighter intersection = closer to true shape.

In [ ]:
def backproject_silhouettes(masks_dict, angles, resolution):
    """
    Visual Hull reconstruction from 2D silhouette masks.
    
    For each angle:
      - Extrude the 2D mask along Y to form a 3D slab
      - Rotate the slab back by -angle
    Intersect all slabs → Visual Hull.
    
    Parameters
    ----------
    masks_dict : {angle: 2D binary np.array}
    angles     : list of angles in degrees
    resolution : voxel grid side length
    
    Returns
    -------
    volume : (R, R, R) binary float32 array
    """
    volume = np.ones((resolution, resolution, resolution), dtype=np.float32)

    for angle in angles:
        mask = masks_dict[angle]   # (X, Z)
        # Extrude: repeat mask along Y axis → (X, Y, Z)
        extruded = np.repeat(mask[:, np.newaxis, :], resolution, axis=1)
        # Rotate back by -angle around Z-axis
        back = rotate(extruded, -angle, axes=(0, 1),
                      reshape=False, order=1)
        # Intersect with accumulator
        volume *= (back > 0.3).astype(np.float32)

    # Post-process: fill holes, remove noise
    volume = binary_fill_holes(volume > 0.5).astype(np.float32)
    volume = morphology.remove_small_objects(
                 volume.astype(bool), max_size=200).astype(np.float32)
    return volume


# ── Reconstruct for all methods ───────────────────────────────
print('Reconstructing 3D volumes …')
reconstructions = {}

for method_name in METHODS:
    print(f'  [{method_name}] …', end=' ')
    vol = backproject_silhouettes(segmented[method_name], ANGLES, RESOLUTION)
    reconstructions[method_name] = vol
    pv   = int(vol.sum())
    vd   = abs(pv - GT_VOLUME) / GT_VOLUME * 100
    p, g = vol.astype(bool), ground_truth.astype(bool)
    dice = 2*(p&g).sum()/(p.sum()+g.sum()+1e-9)
    print(f'Voxels={pv:,}  VolDiff={vd:.2f}%  Dice={dice:.4f}')

print('\nReconstruction done ✓')

---
## 6 · Step 4 — Display Reconstructed 3D (Rotatable Plotly)

> Drag to rotate · Scroll to zoom · Right-drag to pan

In [ ]:
def extract_mesh(volume, step_size=2):
    try:
        v, f, _, _ = measure.marching_cubes(
            volume.astype(np.float32), level=0.5,
            step_size=step_size, allow_degenerate=False)
        return v, f
    except Exception as e:
        print(f'  Mesh failed: {e}'); return None, None

def volume_difference(pred, gt):
    pv, gv = int(pred.sum()), int(gt.sum())
    return pv, gv, abs(pv - gv) / (gv + 1e-9) * 100

def dice_score(pred, gt):
    p, g = pred.astype(bool), gt.astype(bool)
    return 2*(p&g).sum()/(p.sum()+g.sum()+1e-9)

CAMERA_ANGLES = {
    'Anterior (0°)'  : dict(x= 0.0, y= 2.5, z=0.4),
    'Posterior (180°)': dict(x= 0.0, y=-2.5, z=0.4),
    'Lateral (90°)'  : dict(x= 2.5, y= 0.0, z=0.4),
    'Superior'       : dict(x= 0.0, y= 0.3, z=2.8),
}

def plot_reconstruction_plotly(volume, name, vd_pct=None, dice_val=None):
    """
    Rotatable Plotly Mesh3d with 4 angle-snap buttons.
    Shows the 3D reconstruction from 4 projection images.
    """
    verts, faces = extract_mesh(volume)
    if verts is None: return

    color = PALETTE.get(name, '#90caf9')
    pv    = int(volume.sum())

    if vd_pct is None:
        subtitle = f'Voxels: {pv:,}  |  Vol Diff: —  |  Dice: 1.0000'
    else:
        subtitle = (f'Voxels: {pv:,}  |  Vol Diff: {vd_pct:.2f}%'
                    f'  |  Dice: {dice_val:.4f}  |  Reconstructed from 4 views')

    buttons = [dict(
        label=aname, method='relayout',
        args=[{'scene.camera': dict(eye=eye, up=dict(x=0,y=0,z=1))}]
    ) for aname, eye in CAMERA_ANGLES.items()]

    fig = go.Figure(data=[go.Mesh3d(
        x=verts[:,0], y=verts[:,1], z=verts[:,2],
        i=faces[:,0], j=faces[:,1], k=faces[:,2],
        color=color, opacity=0.90, flatshading=False,
        lighting=dict(ambient=0.25, diffuse=0.85, specular=0.35,
                      roughness=0.45, fresnel=0.2),
        lightposition=dict(x=200, y=300, z=200),
    )])

    fig.update_layout(
        title=dict(
            text=f'<b>{name}</b><br><sup>{subtitle}</sup>',
            font=dict(size=13, color='white'), x=0.5, xanchor='center'
        ),
        paper_bgcolor='#0d0d1a',
        scene=dict(
            bgcolor='#0d0d1a',
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            camera=dict(eye=CAMERA_ANGLES['Anterior (0°)'],
                        up=dict(x=0, y=0, z=1)),
            aspectmode='data'
        ),
        updatemenus=[dict(
            type='buttons', direction='right',
            x=0.5, xanchor='center', y=0.02, yanchor='bottom',
            bgcolor='#1e1e3a', font=dict(color='white', size=11),
            buttons=buttons, showactive=True, active=0
        )],
        annotations=[dict(
            text='<b>View:</b>', showarrow=False,
            x=0.1, y=0.02, xref='paper', yref='paper',
            font=dict(color='#aaa', size=11)
        )],
        margin=dict(l=0, r=0, t=80, b=60),
        width=750, height=580
    )
    fig.show()
    print(f'  [{name}] Verts={len(verts):,}  Faces={len(faces):,}')

print('Plotly plotter defined ✓')

In [ ]:
# ── Ground Truth 3D ───────────────────────────────────────────
plot_reconstruction_plotly(ground_truth, 'Ground Truth')

In [ ]:
# ── Method 1: Otsu ───────────────────────────────────────────
r = reconstructions['Otsu']
_, _, vd = volume_difference(r, ground_truth)
dc = dice_score(r, ground_truth)
plot_reconstruction_plotly(r, 'Otsu', vd, dc)

In [ ]:
# ── Method 2: Adaptive Threshold ─────────────────────────────
r = reconstructions['Adaptive Threshold']
_, _, vd = volume_difference(r, ground_truth)
dc = dice_score(r, ground_truth)
plot_reconstruction_plotly(r, 'Adaptive Threshold', vd, dc)

In [ ]:
# ── Method 3: Watershed ───────────────────────────────────────
r = reconstructions['Watershed']
_, _, vd = volume_difference(r, ground_truth)
dc = dice_score(r, ground_truth)
plot_reconstruction_plotly(r, 'Watershed', vd, dc)

In [ ]:
# ── Method 4: Chan-Vese ───────────────────────────────────────
r = reconstructions['Chan-Vese']
_, _, vd = volume_difference(r, ground_truth)
dc = dice_score(r, ground_truth)
plot_reconstruction_plotly(r, 'Chan-Vese', vd, dc)

---
## 7 · Full Pipeline Summary Figure

Shows the complete flow: **4 projection images → segmented masks → reconstructed 3D slices** for all methods.

In [ ]:
# ── Pipeline summary: projections + segmented masks side by side ──
all_methods = list(METHODS.keys())
n_rows = 2 + len(all_methods)   # orig projections + GT binary + each method
row_labels = ['Original\nProjection', 'GT Mask'] + all_methods
row_colors = ['#4e9af1','#4e9af1'] + [PALETTE[m] for m in all_methods]

fig, axes = plt.subplots(n_rows, len(ANGLES),
                          figsize=(14, 4.5 * n_rows), facecolor='#0a0a14')
fig.suptitle('Full Pipeline — 2D Projections & Segmented Masks at 0°, 30°, 60°, 90°',
             color='white', fontsize=13, fontweight='bold', y=1.01)

# Column headers
for col, angle in enumerate(ANGLES):
    axes[0][col].set_title(f'Angle = {angle}°', color='#4e9af1',
                            fontsize=12, fontweight='bold')

for row_i, (label, color) in enumerate(zip(row_labels, row_colors)):
    for col, angle in enumerate(ANGLES):
        ax = axes[row_i][col]
        ax.set_facecolor('#0a0a14')
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values(): spine.set_visible(False)

        if row_i == 0:
            img = projections_gt[angle]
            ax.imshow(img.T, cmap='bone', origin='lower', aspect='auto')
        elif row_i == 1:
            # GT binary mask (projection > otsu_threshold)
            t   = threshold_otsu(projections_gt[angle])
            msk = (projections_gt[angle] > t).astype(float)
            ax.imshow(msk.T, cmap='Blues', origin='lower',
                      aspect='auto', vmin=0, vmax=1)
        else:
            method_name = label
            mask = segmented[method_name][angle]
            orig = projections_gt[angle]
            overlay = np.stack([
                np.clip(orig * 0.35 + mask * 0.65, 0, 1),
                np.clip(orig * 0.35, 0, 1),
                np.clip(orig * 0.35, 0, 1)
            ], axis=-1)
            ax.imshow(overlay.transpose(1, 0, 2), origin='lower', aspect='auto')

    # Row label on left
    axes[row_i][0].set_ylabel(label, color=color,
                               fontsize=10, fontweight='bold',
                               rotation=0, ha='right', va='center',
                               labelpad=55)

plt.tight_layout()
plt.savefig('pipeline_summary.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Saved → pipeline_summary.png')

---
## 8 · Quantitative Evaluation

In [ ]:
method_names = list(METHODS.keys())
vol_diffs, dice_scores, pred_voxels = [], [], []

print('=' * 66)
print(f'{"Method":<22} {"Pred. Voxels":>13} {"Vol. Diff (%)">15} {"Dice":>9}')
print('=' * 66)
print(f'{"Ground Truth":<22} {GT_VOLUME:>13,} {"—":>15} {1.000:>9.4f}')
print('-' * 66)

for nm in method_names:
    r = reconstructions[nm]
    pv, _, vd = volume_difference(r, ground_truth)
    dc = float(dice_score(r, ground_truth))
    vol_diffs.append(vd); dice_scores.append(dc); pred_voxels.append(pv)
    flag = ' ★' if dc == max(dice_scores) else ''
    print(f'{nm:<22} {pv:>13,} {vd:>14.2f}% {dc:>9.4f}{flag}')

print('=' * 66)
best_vd = method_names[int(np.argmin(vol_diffs))]
best_dc = method_names[int(np.argmax(dice_scores))]
print(f'\n  Best Vol. Diff : {best_vd} ({min(vol_diffs):.2f}%)')
print(f'  Best Dice      : {best_dc} ({max(dice_scores):.4f})')

In [ ]:
bar_colors = ['#f06292', '#4caf50', '#ffb74d', '#ce93d8']

fig = make_subplots(rows=1, cols=2,
    subplot_titles=[
        'Volume Difference vs Ground Truth  (lower ↓ is better)',
        'Dice Score vs Ground Truth  (higher ↑ is better)'
    ])

fig.add_trace(go.Bar(
    x=method_names, y=vol_diffs, marker_color=bar_colors,
    text=[f'{v:.1f}%' for v in vol_diffs], textposition='outside'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=method_names, y=dice_scores, marker_color=bar_colors,
    text=[f'{v:.3f}' for v in dice_scores], textposition='outside'
), row=1, col=2)

fig.add_hline(y=1.0, line_dash='dash', line_color='#4e9af1',
              annotation_text='Perfect = 1.0', row=1, col=2)

fig.update_layout(
    title=dict(
        text='Femur 3D Reconstruction from 4 Views — Evaluation',
        font=dict(size=14), x=0.5, xanchor='center'
    ),
    paper_bgcolor='#f5f5f5', plot_bgcolor='#fff',
    showlegend=False, height=450, width=900
)
fig.update_yaxes(range=[0, max(vol_diffs) * 1.3 + 2], row=1, col=1)
fig.update_yaxes(range=[0, 1.15], row=1, col=2)
fig.show()

---
## 9 · Summary

### Pipeline
```
Femur.stl → Voxelize (80³) → Rotate & MIP → 2D images at 0°, 30°, 60°, 90°
                                    ↓ (per method)
                           Segment bone in each 2D image
                                    ↓
                   Back-project + intersect → 3D Visual Hull
```

### Segmentation Methods
| Method | 2D Algorithm | Notes |
|---|---|---|
| **Otsu** | Global threshold (minimise intra-class variance) | Fast, one value per image |
| **Adaptive** | Local block mean threshold (21×21 window) | No single threshold value |
| **Watershed** | Gradient flood from bone/background seeds | Handles touching regions |
| **Chan-Vese** | Variational energy contour | No gradient needed |

### Angles Used
| Angle | Rotation | Projection axis |
|---|---|---|
| **0°** | No rotation | Front view |
| **30°** | 30° around Z | Oblique front-right |
| **60°** | 60° around Z | Oblique side |
| **90°** | 90° around Z | True side view |